# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.9 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

- Lecture 1: `wc -l data/*.csv` in the task 1.1 counts each CVS file's lines into output/file_inventory.txt
- Lecture 2: `profile(name, table)` function in task 1.2 is created and called once per file 
- Lecture 3: `uv venv --seed` created virtual environment for this project. In `complete_pairs`, rows are patients and columns are visits. I chose `axis = 0` to get one mean per visit by collapsing the patient axis.
- Lecture 4: `.loc[a1c_long["visit"] == label, "a1c_pct"]` in task 4.1 selects rows by a boolean label condition and a column by name. 
- Lecture 5: task 3.2 turns the `999` sentinel and reading outside 70-250 into missing values, and the task 5.1 `checks` Series with `assert checks.all()` stops the analysis extract rom being saved if any rule fails. 
- Lecture 6: task 4.4 builds `analysis` with three left merges, so a repeated key raises a `MergeError` instead of silently duplicating visits. 

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
# paste your command here
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [ ]:
#write a function, such as profile(name, table), that returns the four values in README
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.
def profile(name, table):
    return{
        "table": name, 
        "rows": len(table), 
        "columns": table.shape[1], 
        "exact_duplicate_rows": table.duplicated().sum()
        }
results = []
for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

table_profile = pd.DataFrame(results)
table_profile.to_csv(PROFILE_PATH, index = False)
table_profile

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

Grain and key of each cleaned table:

| Cleaned table | One row is | Key |
| --- | --- | --- |
| `patients_clean` | one registered patient | `patient_id` (single) |
| `visits_clean` | one visit of one patient | `patient_id` + `visit` (composite) |
| `sites_current` | one site | `site_code` (single) |
| `a1c_long`  | one A1c result for one patient at one visit | `patient_id` + `visit` (composite) |

Raw rows that break these keys:

- `patients.parquet` has 45 exact duplicate rows, so some `patient_id` values repeat until they are removed.
- The visit files have 35 exact duplicate rows each, and after the visit labels are mapped (`wk12` → `week12`), 240 rows form 120 `patient_id` + `visit` pairs read twice at different times; these rechecks are not exact duplicates, so the later reading is kept.
- `sites.csv` lists `N01` and `S02` twice, once `current` and once `retired`, so `site_code` repeats until only current records are kept; `W03` has no patients, and the registry code `E04` is not in the site list.
- `a1c_wide.parquet` has one row per patient, so after melting each `patient_id` + `visit` pair appears once.

Expected cardinality of the Task 4.4 joins:

- visits → patients on `patient_id`: many-to-one (each patient can have multiple visits).
- patients → sites on `site_code`: many-to-one (many patients share one current site).
- visits → A1c results on `patient_id` + `visit`: one-to-one (one result per patient visit).


### 2.2 Stack the two visit files

In [ ]:
# rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
south = raw["visits_south"].copy(deep = True).rename(columns= {"wt_kg": "weight_kg"})
north = raw["visits_north"].copy(deep = True)

south["source_file"] = "visits_south.csv"
north["source_file"] = "visits_north.csv"

visits_raw = pd.concat([north, south], ignore_index = True)
visits_raw.head()

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


### 2.3 Audit the raw tables

In [ ]:
# count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
# count, and save it to AUDIT_PATH.
patients = raw["patients"]
sites = raw["sites"]
visits = visits_raw

gender_stripped = patients["gender"].str.strip()
site_normalized = patients["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")
sbp = pd.to_numeric(visits["sbp_mmhg"], errors = "coerce")
date_ok = (
    visits["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
    & pd.to_datetime(visits["visit_date"], format = "%Y-%m-%d", errors = "coerce").notna()
)
visit_map = {"baseline": "baseline", "bl": "baseline",
             "week12": "week12", "wk12": "week12", "week 12": "week12"}
visit_mapped = visits["visit"].str.strip().str.lower().map(visit_map)

counts = {
    "patients: gender values that are not a code": ((gender_stripped != "")& ~patients["gender"].isin(["F", "M", "X"])).sum(),
    "patients: empty gender": (gender_stripped == "").sum(),
    "patients: birth_year sentinel 9999": (patients["birth_year"] == "9999").sum(),
    "patients: site codes needing normalization": (site_normalized != patients["site_code"]).sum(),

    "sites: rows with a repeated site_code": sites.duplicated(subset = ["site_code"], keep = False).sum(),
    "visits: visit labels that are not baseline or week12": (~visits["visit"].isin(["baseline", "week12"])).sum(),
    "visits: sbp sentinel 999": (sbp == 999).sum(),
    "visits: sbp outside 70 through 250": ((sbp != 999) & ((sbp<70)|(sbp>250))).sum(), 
    "visits: date parse failures": (~date_ok).sum(), 
    "visits: rows in repeated patient_id and visit pairs": 
    visits.assign(visit=visit_mapped).duplicated(subset = ["patient_id", "visit"], keep = False).sum(), 
}

issue_audit = pd.DataFrame({"issue": list(counts), "count": [int(c) for c in counts.values()]})
issue_audit.to_csv(AUDIT_PATH, index =False)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

## Task 3: Clean

### 3.1 Clean the patients

In [ ]:
# list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.
display(raw["patients"]["gender"].value_counts())
patients_clean = raw["patients"].copy(deep=True)

patients_clean =patients_clean.drop_duplicates(keep="first")

gender_map = {
    "f": "F", "female": "F", "woman": "F",
    "m": "M", "male": "M", "man": "M",
    "x": "X", "nonbinary": "X", "non-binary": "X", "nb": "X",
}
patients_clean["gender"] = patients_clean["gender"].str.strip().str.lower().map(gender_map).astype("category")
patients_clean["birth_year"] = patients_clean["birth_year"].replace("9999", pd.NA).astype("Int64")
patients_clean["site_code"] = patients_clean["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")

patients_clean = patients_clean.reset_index(drop=True)
patients_clean.head()

gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]

,patient_id,gender,birth_year,site_code
0,P00001,F,1966,S02
1,P00002,F,<NA>,N01
2,P00003,F,1970,N01
3,P00004,X,<NA>,N01
4,P00005,M,1947,N01


In [ ]:
# show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.
print(patients_clean["gender"].cat.categories)

patients_summary = pd.DataFrame({"metric": [
        "patients", 
        "birth_year missing", 
        "birth_year min", 
        "birth_year max", 
        "site_code values", 
        "patients at N01", 
        "patients at S02", 
        "patients at E04"
        ],
        "value": [
        len(patients_clean), 
        patients_clean["birth_year"].isna().sum(), 
        patients_clean["birth_year"].min(), 
        patients_clean["birth_year"].max(), 
        patients_clean["site_code"].nunique(), 
        (patients_clean["site_code"]=="N01").sum(),
        (patients_clean["site_code"]=="S02").sum(),
        (patients_clean["site_code"]=="E04").sum()
        ],
})

patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index = False)
display(patients_summary)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)
gender_counts

Index(['F', 'M', 'X'], dtype='str')


,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [8]:
visits_raw

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
...,...,...,...,...,...,...,...
57366,P24976,week12,2026-05-14,10:50,153,78.3,visits_south.csv
57367,P19276,baseline,2026-01-22,15:35,160,122.2,visits_south.csv
57368,P21334,week12,2026-05-05,09:40,129,98.2,visits_south.csv
57369,P03524,baseline,2026-02-06,10:05,160,117.6,visits_south.csv


In [9]:
visit_map

{'baseline': 'baseline',
 'bl': 'baseline',
 'week12': 'week12',
 'wk12': 'week12',
 'week 12': 'week12'}

In [ ]:
# build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.
display(visits_raw["visit"].value_counts())

visits_clean = visits_raw.copy(deep=True)

visits_clean = visits_clean.drop_duplicates(keep="first")
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(visit_map)
sbp_values = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = sbp_values.where(sbp_values.between(70, 250)).astype("Int64")
visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")

exact_date = visits_clean["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
visits_clean["visit_date"] = pd.to_datetime(
    visits_clean["visit_date"].where(exact_date), format="%Y-%m-%d", errors="coerce"
)
visits_clean.head()

visit
baseline     28533
week12       26529
BASELINE       256
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        195
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,week12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


In [ ]:
# step 6, show the repeated patient_id and visit pairs, then keep the later reading.
repeated_pair = visits_clean.duplicated(subset = ["patient_id", "visit"], keep = False)
display(visits_clean[repeated_pair].sort_values(["patient_id", "visit", "visit_time"]))

visits_clean = (
    visits_clean.sort_values("visit_time", kind="stable")
    .drop_duplicates(subset=["patient_id", "visit"], keep="last")
)
print("Repeated pairs left:", visits_clean.duplicated(subset=["patient_id", "visit"]).sum())

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


Repeated pairs left: 0


In [ ]:
# step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).
visits_clean = (
    visits_clean.sort_values(["patient_id", "visit"])
    .reset_index(drop=True)
    [["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]]
)
display(visits_clean["visit"].value_counts(dropna=False))

visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv",
    ],
    "value": [
        len(visits_clean),
        (visits_clean["visit"] == "baseline").sum(),
        (visits_clean["visit"] == "week12").sum(),
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        (visits_clean["source_file"] == "visits_north.csv").sum(),
        (visits_clean["source_file"] == "visits_south.csv").sum(),
    ],
})
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)
visits_summary

visit
baseline    29599
week12      27582
Name: count, dtype: int64

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [ ]:
# one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decisions = [
    ("patients", "exact duplicate rows",
     "Dropped exact duplicate rows, keeping the first copy.",
     "A registry row repeated in every column is the same patient entered twice, so keeping one copy makes patient_id a unique key."),
    ("patients", "gender spellings that are not a code",
     "Stripped and lowercased gender, mapped each spelling to F, M, or X, and stored it as a category.",
     "Spellings such as woman, MALE, M with a trailing space, and non-binary name the same three codes, while empty and unknown entries were never recorded and become missing."),
    ("patients", "site codes needing normalization",
     "Uppercased site_code and removed every hyphen and space.",
     "Entries such as n-01 or S 02 refer to N01 and S02, and the site join needs the exact code to match sites.csv."),
    ("patients", "birth_year sentinel 9999",
     "Replaced 9999 with a missing value and stored birth_year as Int64.",
     "The data dictionary says 9999 means not recorded, so keeping it as a year would distort the minimum, maximum, and any age."),
    ("visits", "weight column named wt_kg in visits_south.csv",
     "Renamed wt_kg to weight_kg in a copy of the south table before stacking.",
     "Both offices record the same weight in kg, and concat aligns columns by name, so a different name would split weight into two half-empty columns."),
    ("visits", "sbp sentinel 999 and readings outside 70 through 250",
     "Set 999 and every reading below 70 or above 250 mmHg to missing, keeping 70 and 250, and stored sbp_mmhg as Int64.",
     "999 means not measured and readings outside 70 through 250 are not plausible systolic pressures, while the planted 70 and 250 values sit on the inclusive range."),
    ("visits", "impossible and loosely written visit dates",
     "Kept visit_date only when the text is exact YYYY-MM-DD for a real calendar date, and set the rest to missing.",
     "Values like 2026/04/27, 2026-3-06, or a day that does not exist on the calendar cannot be trusted as written, so guessing a date would invent data."),
    ("visits", "repeated patient_id and visit pairs",
     "Mapped the visit labels, removed exact repeats, then kept the later reading of each remaining pair by visit_time.",
     "After mapping, labels such as wk12 and week12 name the same visit, and a second reading at a later time is a recheck that should replace the first."),
    ("sites", "repeated site_code rows",
     "Kept only rows whose record_status is current, with site_code and site_name.",
     "N01 and S02 each have a retired and a current record, and keeping both would duplicate every patient row in the site join."),
]
decision_table = pd.DataFrame(decisions, columns=["table", "issue", "decision", "reason"])
decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,"Dropped exact duplicate rows, keeping the firs...",A registry row repeated in every column is the...
1,patients,gender spellings that are not a code,"Stripped and lowercased gender, mapped each sp...","Spellings such as woman, MALE, M with a traili..."
2,patients,site codes needing normalization,Uppercased site_code and removed every hyphen ...,Entries such as n-01 or S 02 refer to N01 and ...
3,patients,birth_year sentinel 9999,Replaced 9999 with a missing value and stored ...,The data dictionary says 9999 means not record...
4,visits,weight column named wt_kg in visits_south.csv,Renamed wt_kg to weight_kg in a copy of the so...,"Both offices record the same weight in kg, and..."
5,visits,sbp sentinel 999 and readings outside 70 throu...,Set 999 and every reading below 70 or above 25...,999 means not measured and readings outside 70...
6,visits,impossible and loosely written visit dates,Kept visit_date only when the text is exact YY...,"Values like 2026/04/27, 2026-3-06, or a day th..."
7,visits,repeated patient_id and visit pairs,"Mapped the visit labels, removed exact repeats...","After mapping, labels such as wk12 and week12 ..."
8,sites,repeated site_code rows,"Kept only rows whose record_status is current,...",N01 and S02 each have a retired and a current ...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [ ]:
# melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = raw["a1c_wide"].melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct",
)
a1c_long["visit"] = a1c_long["visit"].map({"a1c_baseline": "baseline", "a1c_week12": "week12"})

print("a1c_pct dtype:", a1c_long["a1c_pct"].dtype)
assert a1c_long["a1c_pct"].dtype == "float64"
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

a1c_summary = pd.DataFrame({
    "visit": ["baseline", "week12"],
    "results": [(a1c_long["visit"] == label).sum() for label in ["baseline", "week12"]],
    "a1c_pct_mean": [
        round(a1c_long.loc[a1c_long["visit"] == label, "a1c_pct"].mean(), 2)
        for label in ["baseline", "week12"]
    ],
})
a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)
a1c_summary

a1c_pct dtype: float64


,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [ ]:
# steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]

try:
    patients_clean.merge(sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)

display(sites[sites.duplicated(subset=["site_code"], keep=False)])

sites_current = sites.loc[sites["record_status"] == "current", ["site_code", "site_name"]]
sites_current = sites_current.reset_index(drop=True)
sites_current

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
1,S02,South Valley Clinic
2,W03,West Clinic


In [ ]:
# step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.
visits_a1c = visits_clean.merge(a1c_long, on=["patient_id", "visit"], how="outer", indicator=True)
merge_counts = visits_a1c["_merge"].value_counts()
display(merge_counts)

merge_audit = pd.DataFrame({
    "check": [
        "visits with an A1c result",
        "visits without an A1c result",
        "A1c results without a visit",
        "visits whose patient is not in the registry",
        "registry patients with no visit",
        "registry patients whose site is not in the site list",
        "site list rows removed by the current-record rule",
    ],
    "count": [
        merge_counts["both"],
        merge_counts["left_only"],
        merge_counts["right_only"],
        (~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum(),
        (~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum(),
        (~patients_clean["site_code"].isin(sites["site_code"])).sum(),
        len(sites) - len(sites_current),
    ],
})
merge_audit["count"] = merge_audit["count"].astype(int)
merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)
merge_audit

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64

,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [ ]:
# pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = visits_clean.pivot(index="patient_id", columns="visit", values="sbp_mmhg").reset_index()
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]

sbp_wide_sample = sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)]
sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index=False)
sbp_wide_sample

,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
3,P00004,142,<NA>,<NA>
87,P00088,152,147,-5
259,P00260,70,138,68
445,P00446,250,144,-106
486,P00487,<NA>,143,<NA>
849,P00851,<NA>,129,<NA>
2336,P02354,127,125,-2
3789,P03836,134,125,-9
5684,P05763,161,151,-10


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [ ]:
# select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.
complete_pairs = sbp_wide[["baseline", "week12"]].dropna()
bp_array = np.array(complete_pairs, dtype=float)
print("shape (patients, visits):", bp_array.shape)

# Rows are patients and columns are visits, so axis=0 averages down the patients for each visit.
visit_means = bp_array.mean(axis=0)
print("mean SBP (mmHg): baseline", visit_means[0].round(2), "| week12", visit_means[1].round(2))

shape (patients, visits): (27149, 2)
mean SBP (mmHg): baseline 143.01 | week12 137.15


### 4.4 Join the analysis table

In [ ]:
# three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = (
    visits_clean
    .merge(patients_clean, on="patient_id", how="left", validate="many_to_one")
    .merge(sites_current, on="site_code", how="left", validate="many_to_one")
    .merge(a1c_long, on=["patient_id", "visit"], how="left", validate="one_to_one")
)

analysis_dtypes = {
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",
}
analysis = analysis[list(analysis_dtypes)].astype(analysis_dtypes)
print(len(visits_clean), "clean visits ->", len(analysis), "analysis rows")
analysis.dtypes

57181 clean visits -> 57181 analysis rows


patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [ ]:
# collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.
checks = pd.Series({
    "analysis has one row per clean visit": len(analysis) == len(visits_clean),
    "no repeated patient_id and visit pair": not analysis.duplicated(subset=["patient_id", "visit"]).any(),
    "sbp_mmhg present values are 70 through 250": analysis["sbp_mmhg"].dropna().between(70, 250).all(),
    "site_name present values are current names": analysis["site_name"].dropna().isin(sites_current["site_name"]).all(),
    "sbp_wide patient_id is unique": sbp_wide["patient_id"].is_unique,
    "raw tables equal their snapshots": all(raw[name].equals(raw_snapshot[name]) for name in raw),
}).astype(bool)
display(checks)

assert checks.all(), checks[~checks]

analysis has one row per clean visit          True
no repeated patient_id and visit pair         True
sbp_mmhg present values are 70 through 250    True
site_name present values are current names    True
sbp_wide patient_id is unique                 True
raw tables equal their snapshots              True
dtype: bool

### 5.2 Save the analysis extract

In [ ]:
# keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).
analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)
print(len(analysis_sample), "rows; expected", visits_clean["patient_id"].isin(SAMPLE_IDS).sum())
analysis_sample

28 rows; expected 28


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [ ]:
# read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.
read_dtypes = {column: dtype for column, dtype in analysis_dtypes.items() if column != "visit_date"}
round_trip = pd.read_csv(ANALYSIS_SAMPLE_PATH, dtype=read_dtypes, parse_dates=["visit_date"])
print(round_trip.equals(analysis_sample))

True
